# Visualize results

This is a notebook to visualize how many checkpoints it takes per model for linear probes to "learn" the dataset (for early stopping to engage).

The reason to look at this is both to get an idea of if we've trained enough, but also to get insight into a) the potential difficulty of a task, or b) the expressive power of the embeddings.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import warnings

# This will ignore all UserWarning messages
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
import polars as pl
# from pathlib import Path
import plotly.express as px

from project.utils.strs import linear_probe_model_dir, linear_probe_results_dir
from project.utils.functions import load_config


In [ ]:
# Make dirs if they don't exist already
for directory in [linear_probe_model_dir, linear_probe_results_dir]:
    directory.mkdir(parents=True, exist_ok=True)

### Getting info on checkpoints of saved models

In [ ]:
model_dir = linear_probe_model_dir

# Storage structure
checkpoint_info = []
# Get files that have terminal root with epoch checkpoint
for f in model_dir.rglob('*epoch_*.ckpt'):
    try:
        fname = str(f)
        if 'normalized' not in fname:
            layer_num = int(fname.split('layer_')[-1].split('/')[0])
            num_epochs = int(fname.split('epoch_')[-1].split('.ckpt')[0])
            dataset_name = fname.split('/')[-3].split('_facebook')[0].split('_chandar')[0].split('_Lolalb')[0]
            model_name = fname.split('_layer_')[0].split(dataset_name + '_')[-1]
            info = {
                'dataset': dataset_name,
                'model_name': model_name,
                'normal_state': False,
                'layer_num': layer_num,
                'num_epochs_best': num_epochs,
            }
            checkpoint_info.append(info)

        elif 'normalized' in fname:
            normal_state = bool(fname.split('_normalized_')[-1].split('/')[0])
            layer_num = int(fname.split('layer_')[-1].split('_normalized')[0])
            num_epochs = int(fname.split('epoch_')[-1].split('.ckpt')[0])
            dataset_name = fname.split('/')[-3].split('_facebook')[0].split('_chandar')[0].split('_Lolalb')[0]
            model_name = fname.split('_layer_')[0].split(dataset_name + '_')[-1]
            info = {
                'dataset': dataset_name,
                'model_name': model_name,
                'normal_state': normal_state,
                'layer_num': layer_num,
                'num_epochs_best': num_epochs,
            }
            checkpoint_info.append(info)

    except Exception as e:
        # print(e)
        # print(fname)
        continue
    

Combining all the information:

In [ ]:
checkpoints_df = pl.DataFrame(checkpoint_info).sort(by=['dataset', 'model_name', 'layer_num'])
checkpoints_df

In [ ]:
# Add dataset size info
dataset_sizes = []
for dataset in checkpoints_df['dataset'].unique():
    df = pl.read_parquet(load_config(dataset)['data_path'])
    dataset_sizes.append(
        {'dataset': dataset,
        'n_samples': len(df),
        'n_train': len(df.filter(pl.col('split')=='train')),
        }
    )
checkpoints_df = checkpoints_df.join(pl.DataFrame(dataset_sizes), on='dataset', how='left')

In [ ]:
# Export to results
checkpoints_df.write_parquet(linear_probe_results_dir / 'linear_probing_model_best_checkpoints.parquet.gz')

### Visualizing the checkpointing

Original datasets: split1, all protein lengths, not normalizing plm embeddings before probing

How this changes if model embeddings are normalized before probe training:

In [ ]:
fig = px.line(checkpoints_df.filter(~pl.col('dataset').str.contains('512') & ~pl.col('dataset').str.contains('split') & pl.col('normal_state')),
x = 'layer_num',
y='num_epochs_best',
facet_col='dataset',
facet_col_wrap = 4,
width = 1500,
height=1100,
color='model_name', labels={'num_epochs_best': 'n_epch_best_ckpt'})
fig.show()

Generally, it seems to be harder to train probes on the normalized embeddings, suggesting that there's valuable signal for most datasets.

In [ ]:
fig = px.line(checkpoints_df.filter(pl.col('dataset').str.contains('512')),
x = 'layer_num',
y='num_epochs_best',
facet_col='dataset',
facet_col_wrap = 4,
width = 1500,
height=1100,
color='model_name', labels={'num_epochs_best': 'n_epch_best_ckpt'})
fig.show()

Very similar behaviour for datasets with only proteins <512 aas, regardless of the model (includes model checkpoints!)

### Correlations

Looking at whether there's a relationship between number of epochs to 'best' probing performance and how much training data is available

In [ ]:
# Basic Pearson Correlation (Linear relationship)
pearson_corr = checkpoints_df.filter(~pl.col('normal_state')).select([
    pl.corr("num_epochs_best", "n_train").alias("corr_n_train"),
    pl.corr("num_epochs_best", "layer_num").alias("corr_layers"),
])

# Spearman Correlation (Better for non-linear/ordinal trends)
spearman_corr = checkpoints_df.filter(~pl.col('normal_state')).select([
    pl.corr("num_epochs_best", "n_train", method="spearman").alias("spearman_n_train"),
    pl.corr("num_epochs_best", "layer_num", method="spearman").alias("spearman_layers"),
])

print("Pearson Correlations:")
print(pearson_corr)

print("\nSpearman Correlations:")
print(spearman_corr)

Checking if the same is true for when normal_state is true:

In [ ]:
# Basic Pearson Correlation (Linear relationship)
pearson_corr = checkpoints_df.filter(pl.col('normal_state')).select([
    pl.corr("num_epochs_best", "n_train").alias("corr_n_train"),
    pl.corr("num_epochs_best", "layer_num").alias("corr_layers"),
])

# Spearman Correlation (Better for non-linear/ordinal trends)
spearman_corr = checkpoints_df.filter(pl.col('normal_state')).select([
    pl.corr("num_epochs_best", "n_train", method="spearman").alias("spearman_n_train"),
    pl.corr("num_epochs_best", "layer_num", method="spearman").alias("spearman_layers"),
])

print("Pearson Correlations:")
print(pearson_corr)

print("\nSpearman Correlations:")
print(spearman_corr)

Conclusions:
- Smaller datasets were harder to get signal for (see: interpro active site, binding site, repeat). This is backed up by a negative Pearson and Spearman correlation: as training set size increases, number of epochs to reach "best" model decreases.
- Generally, bigger models had lower numbers of epochs than smaller, suggesting there's more representational power in them and it's 'easier' to learn useful patterns.
- Lower correlations with normal_state=True suggests that it's a bit harder for probes to get signal, even as training set size increases.